# 第2回 発展：用語集Q&AをLoRAで学習し、辞書検索と比較する

公式ノートブックから作った発展課題用の実行版です。学習結果はまだ入っていません。
**T4 GPUを選び、セクション0〜7を上から実行してください。** GPUを使った学習は本人が行い、出力を目視で確認して考察を書きます。

変更した箇所：
- セクション2：公式 `make_sft_dataset.py` で用語集58語から116件のQ&Aを作成。
- セクション3〜5：FAQを入力に渡す実験から、質問のみで用語を説明する実験へ変更。回答は平文なので、元のJSON形式の採点は使いません。
- セクション6：同じ24問を、正規名・別名の完全一致による辞書検索でも評価。
- セクション7：設定、学習時間、最終loss、学習前後と検索の回答、評価データを保存。

評価は既知語10問・別名/言い換え10問・用語集外4問です。質問は学習・検証に使った質問と重複しません。既知語の定義自体は学習に含まれるので、これは未見の聞き方への対応を見る実験です。
自動の要点一致率は参考指標です。否定・矛盾・説明不足は検出できないので、回答全文も確認してください。

公式課題：https://github.com/matsuolab/lecture-ai-engineering-2026/blob/main/lectures/02/problem.md
公式ノートブック：https://github.com/matsuolab/lecture-ai-engineering-2026/blob/main/notebooks/lora_format_tuning.ipynb
用語集blob：bc0d6807bdb9aeef7ef9eb98e74d767e056ed745
生成スクリプトblob：7e53cfc38f0b70f0141b6f98eab18a94c16a2645

## 0. 環境の準備
公式ノートブックと同じ依存バージョンを使います。最初にT4が表示されることを確認してください。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定と保存先
`USE_DRIVE = True` なら、このセルを実行したときにDrive接続の確認が出ます。
`MyDrive/aie2026/lecture02_advanced/実行日時/` へ結果を保存します。

LoRAの `lora_r=16`、`lr=2e-4`、モデルとrevisionは公式版と同じです。
学習データが14件から116件になるため、まず `num_train_epochs=3` としています。1エポック約15ステップ、合計約45ステップです。
学習が終わったら時間とlossを確認し、追加実験では条件を記録してください。実行日時ごとに保存先が分かれます。

In [3]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json, os, time, shutil, random

RUN_ID = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d_%H%M%S_%f')
CONFIG = {
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'revision': '2fc06364715b967f1860aea9cf38778875588b17',
    'seed': 42,
    'num_train_epochs': 3,
    'warmup_steps': 6,
    'lr': 2e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'max_seq_len': 1024,
    'max_new_tokens': 256,
    'gen': {'do_sample': False},
    'work_dir': f'/content/lecture02_advanced/{RUN_ID}',
    'drive_dir': f'/content/drive/MyDrive/aie2026/lecture02_advanced/{RUN_ID}',
}
USE_DRIVE = True
WORK = Path(CONFIG['work_dir'])
WORK.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    Path(CONFIG['drive_dir']).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 3,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 1024,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/lecture02_advanced/20261102_160734_926704",
  "drive_dir": "/content/drive/MyDrive/aie2026/lecture02_advanced/20261102_160734_926704"
}


## 2. 用語集データと評価セット
このセルは公式の用語集と生成スクリプトのスナップショットをファイルに書き、スクリプトを実行します。
外部ファイルの手動アップロードは不要です。

`glossary_sft.jsonl` は公式スクリプトの出力116件、`valid.jsonl` は別の聞き方58件です。
評価の正解・チェック項目は学習前に固定し、モデル入力には渡しません。
用語集外には実在語2つと、この実験用の架空語2つを含めます。実在未収録語の正しい説明を「幻覚」と扱わないでください。

In [4]:
import subprocess, sys, unicodedata, re

SOURCE = WORK / 'source'
(SOURCE / 'data').mkdir(parents=True, exist_ok=True)
(SOURCE / 'scripts').mkdir(parents=True, exist_ok=True)
(SOURCE / 'data/glossary.json').write_text('{\n  "version": "2026-draft-1",\n  "note": "AIE2026 第1〜3回・第5回の用語集（たたき台）。講義スライド確定後に内容を差し替える。",\n  "terms": [\n    {\n      "id": "ai-engineering",\n      "term": "AIエンジニアリング",\n      "aliases": ["AI Engineering"],\n      "lecture": 1,\n      "category": "全体像",\n      "definition": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。",\n      "related": ["llm", "evaluation-loop"]\n    },\n    {\n      "id": "evaluation-loop",\n      "term": "評価と改善のサイクル",\n      "aliases": ["改善サイクル"],\n      "lecture": 1,\n      "category": "全体像",\n      "definition": "作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。",\n      "related": ["ai-engineering", "hallucination"]\n    },\n    {\n      "id": "llm",\n      "term": "LLM（大規模言語モデル）",\n      "aliases": ["大規模言語モデル", "Large Language Model"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。",\n      "related": ["token", "prompt", "pretraining"]\n    },\n    {\n      "id": "token",\n      "term": "トークン",\n      "aliases": ["token"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。",\n      "related": ["context-window", "rate-limit"]\n    },\n    {\n      "id": "context-window",\n      "term": "コンテキストウィンドウ",\n      "aliases": ["コンテキスト長", "context window"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。",\n      "related": ["token", "agent-loop"]\n    },\n    {\n      "id": "prompt",\n      "term": "プロンプト",\n      "aliases": ["prompt"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。",\n      "related": ["system-prompt", "llm"]\n    },\n    {\n      "id": "system-prompt",\n      "term": "システムプロンプト",\n      "aliases": ["指示文", "instructions", "system instruction"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。",\n      "related": ["prompt", "guardrail"]\n    },\n    {\n      "id": "temperature",\n      "term": "temperature",\n      "aliases": ["温度パラメータ"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。",\n      "related": ["llm", "evaluation-loop"]\n    },\n    {\n      "id": "hallucination",\n      "term": "ハルシネーション",\n      "aliases": ["幻覚", "hallucination"],\n      "lecture": 1,\n      "category": "LLM基礎",\n      "definition": "LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。",\n      "related": ["tool", "evaluation-loop"]\n    },\n    {\n      "id": "api",\n      "term": "API",\n      "aliases": ["Application Programming Interface", "Web API"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。",\n      "related": ["api-key", "rate-limit"]\n    },\n    {\n      "id": "api-key",\n      "term": "APIキー",\n      "aliases": ["API key"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。",\n      "related": ["codespaces-secret", "environment-variable"]\n    },\n    {\n      "id": "environment-variable",\n      "term": "環境変数",\n      "aliases": ["environment variable", "env"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。",\n      "related": ["api-key", "codespaces-secret"]\n    },\n    {\n      "id": "rate-limit",\n      "term": "レート制限",\n      "aliases": ["rate limit", "429エラー"],\n      "lecture": 1,\n      "category": "開発基礎",\n      "definition": "一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。",\n      "related": ["api", "agent-loop", "default-model"]\n    },\n    {\n      "id": "codespaces",\n      "term": "GitHub Codespaces",\n      "aliases": ["Codespaces", "コードスペース"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。",\n      "related": ["devcontainer", "codespaces-secret"]\n    },\n    {\n      "id": "devcontainer",\n      "term": "devcontainer",\n      "aliases": ["Dev Container", "devcontainer.json"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。",\n      "related": ["codespaces"]\n    },\n    {\n      "id": "codespaces-secret",\n      "term": "Codespaces secret",\n      "aliases": ["Codespacesのシークレット"],\n      "lecture": 1,\n      "category": "開発環境",\n      "definition": "GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。",\n      "related": ["api-key", "environment-variable", "codespaces"]\n    },\n    {\n      "id": "fork",\n      "term": "fork",\n      "aliases": ["フォーク"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。",\n      "related": ["branch", "pull-request"]\n    },\n    {\n      "id": "branch",\n      "term": "ブランチ",\n      "aliases": ["branch"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。",\n      "related": ["pull-request", "fork"]\n    },\n    {\n      "id": "pull-request",\n      "term": "プルリクエスト（PR）",\n      "aliases": ["PR", "Pull Request"],\n      "lecture": 1,\n      "category": "Git/GitHub",\n      "definition": "あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。",\n      "related": ["branch", "answer-md"]\n    },\n    {\n      "id": "github-copilot",\n      "term": "GitHub Copilot",\n      "aliases": ["Copilot"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。",\n      "related": ["code-completion", "coding-agent"]\n    },\n    {\n      "id": "code-completion",\n      "term": "コード補完",\n      "aliases": ["インライン補完", "inline suggestion"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。",\n      "related": ["github-copilot"]\n    },\n    {\n      "id": "coding-agent",\n      "term": "コーディングエージェント",\n      "aliases": ["Coding Agent", "AIコーディングエージェント"],\n      "lecture": 1,\n      "category": "AIコーディング",\n      "definition": "ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。",\n      "related": ["ai-agent", "harness"]\n    },\n    {\n      "id": "pretraining",\n      "term": "事前学習",\n      "aliases": ["pre-training", "プレトレーニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。",\n      "related": ["fine-tuning", "llm"]\n    },\n    {\n      "id": "fine-tuning",\n      "term": "ファインチューニング",\n      "aliases": ["fine-tuning", "追加学習", "FT"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。",\n      "related": ["sft", "peft", "lora"]\n    },\n    {\n      "id": "sft",\n      "term": "教師ありファインチューニング（SFT）",\n      "aliases": ["SFT", "Supervised Fine-Tuning", "指示チューニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。",\n      "related": ["fine-tuning", "lora"]\n    },\n    {\n      "id": "peft",\n      "term": "PEFT",\n      "aliases": ["Parameter-Efficient Fine-Tuning", "パラメータ効率の良いファインチューニング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。",\n      "related": ["lora", "fine-tuning"]\n    },\n    {\n      "id": "lora",\n      "term": "LoRA",\n      "aliases": ["Low-Rank Adaptation"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。",\n      "related": ["peft", "lora-rank", "qlora"]\n    },\n    {\n      "id": "lora-rank",\n      "term": "ランク（LoRAのr）",\n      "aliases": ["rank", "r"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。",\n      "related": ["lora"]\n    },\n    {\n      "id": "qlora",\n      "term": "QLoRA",\n      "aliases": ["Quantized LoRA"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。",\n      "related": ["lora", "quantization"]\n    },\n    {\n      "id": "quantization",\n      "term": "量子化",\n      "aliases": ["quantization"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。",\n      "related": ["qlora"]\n    },\n    {\n      "id": "learning-rate",\n      "term": "学習率",\n      "aliases": ["learning rate", "lr"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。",\n      "related": ["epoch", "overfitting"]\n    },\n    {\n      "id": "epoch",\n      "term": "エポック",\n      "aliases": ["epoch"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。",\n      "related": ["overfitting", "learning-rate"]\n    },\n    {\n      "id": "overfitting",\n      "term": "過学習",\n      "aliases": ["overfitting", "オーバーフィッティング"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。",\n      "related": ["epoch", "evaluation-loop"]\n    },\n    {\n      "id": "huggingface-hub",\n      "term": "Hugging Face Hub",\n      "aliases": ["Hugging Face", "HF Hub"],\n      "lecture": 2,\n      "category": "ファインチューニング",\n      "definition": "公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。",\n      "related": ["lora", "pretraining"]\n    },\n    {\n      "id": "ai-agent",\n      "term": "AIエージェント",\n      "aliases": ["エージェント", "agent"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。",\n      "related": ["agent-loop", "tool", "harness"]\n    },\n    {\n      "id": "agent-loop",\n      "term": "エージェントループ",\n      "aliases": ["agent loop", "ツール呼び出しループ"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。",\n      "related": ["ai-agent", "tool", "rate-limit"]\n    },\n    {\n      "id": "tool",\n      "term": "ツール（Tool use / Function calling）",\n      "aliases": ["ツール呼び出し", "Function calling", "Tool use", "ツール"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。",\n      "related": ["tool-description", "agent-loop"]\n    },\n    {\n      "id": "tool-description",\n      "term": "ツールの説明文",\n      "aliases": ["docstring", "ツールスキーマ", "tool schema"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。",\n      "related": ["tool", "pydantic-ai"]\n    },\n    {\n      "id": "structured-output",\n      "term": "構造化出力",\n      "aliases": ["structured output", "JSON出力"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。",\n      "related": ["pydantic-ai"]\n    },\n    {\n      "id": "harness",\n      "term": "ハーネス",\n      "aliases": ["agent harness", "エージェントハーネス"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。",\n      "related": ["coding-agent", "pi"]\n    },\n    {\n      "id": "pi",\n      "term": "Pi",\n      "aliases": ["Pi coding agent", "pi agent"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。",\n      "related": ["harness", "coding-agent"]\n    },\n    {\n      "id": "pydantic-ai",\n      "term": "Pydantic AI",\n      "aliases": ["pydantic-ai", "PydanticAI"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。",\n      "related": ["test-model", "structured-output", "adk"]\n    },\n    {\n      "id": "adk",\n      "term": "ADK（Agent Development Kit）",\n      "aliases": ["ADK", "Agent Development Kit", "google-adk"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深い。2.0で1.x系から大きな変更が入っている。本講座では第3回の宿題の発展レベルで扱う。",\n      "related": ["pydantic-ai", "ai-agent"]\n    },\n    {\n      "id": "mcp",\n      "term": "MCP（Model Context Protocol）",\n      "aliases": ["MCP", "Model Context Protocol"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。",\n      "related": ["tool", "ai-agent", "mcp-server", "mcp-client"]\n    },\n    {\n      "id": "guardrail",\n      "term": "ガードレール",\n      "aliases": ["guardrail"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。",\n      "related": ["human-in-the-loop", "system-prompt"]\n    },\n    {\n      "id": "human-in-the-loop",\n      "term": "Human-in-the-loop",\n      "aliases": ["HITL", "人間による承認", "確認を挟む"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。",\n      "related": ["guardrail", "tool"]\n    },\n    {\n      "id": "test-model",\n      "term": "TestModel",\n      "aliases": ["テスト用モデル"],\n      "lecture": 3,\n      "category": "エージェント",\n      "definition": "Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。",\n      "related": ["pydantic-ai", "rate-limit"]\n    },\n    {\n      "id": "homework-levels",\n      "term": "宿題の3レベル",\n      "aliases": ["3レベル", "宿題レベル"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。",\n      "related": ["answer-md"]\n    },\n    {\n      "id": "answer-md",\n      "term": "answer.md",\n      "aliases": ["解答ファイル"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。",\n      "related": ["pull-request", "homework-levels"]\n    },\n    {\n      "id": "lec-start-tag",\n      "term": "lecNN-start タグ",\n      "aliases": ["開始タグ", "lec03-start"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。",\n      "related": ["branch"]\n    },\n    {\n      "id": "default-model",\n      "term": "既定モデル",\n      "aliases": ["デフォルトモデル", "gemini-3.5-flash-lite"],\n      "lecture": 1,\n      "category": "講座",\n      "definition": "AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。",\n      "related": ["rate-limit", "api-key"]\n    },\n    {\n      "id": "external-tool",\n      "term": "外部ツール",\n      "aliases": ["external tool", "外部サービス"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。",\n      "related": ["tool", "mcp", "mcp-server"]\n    },\n    {\n      "id": "mcp-server",\n      "term": "MCPサーバー",\n      "aliases": ["MCP server"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。",\n      "related": ["mcp", "mcp-client", "mcp-transport", "fetch-server"]\n    },\n    {\n      "id": "mcp-client",\n      "term": "MCPクライアント",\n      "aliases": ["MCP client", "MCPホスト"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。",\n      "related": ["mcp", "mcp-server", "mcp-config"]\n    },\n    {\n      "id": "mcp-transport",\n      "term": "トランスポート（stdio・Streamable HTTP）",\n      "aliases": ["stdio", "Streamable HTTP", "transport", "MCPのトランスポート"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。",\n      "related": ["mcp-server", "mcp-client"]\n    },\n    {\n      "id": "mcp-config",\n      "term": "mcpServers設定",\n      "aliases": ["mcpServers", "MCPの設定ファイル", "mcp_servers.json", "mcp.json"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。",\n      "related": ["mcp-client", "tool-namespace"]\n    },\n    {\n      "id": "tool-namespace",\n      "term": "ツールの名前空間",\n      "aliases": ["ツール名のプレフィックス", "prefix", "名前の衝突"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。",\n      "related": ["mcp-config", "tool-description", "rate-limit"]\n    },\n    {\n      "id": "fetch-server",\n      "term": "fetchサーバー",\n      "aliases": ["mcp-server-fetch", "fetch", "Web取得ツール"],\n      "lecture": 5,\n      "category": "MCP",\n      "definition": "指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。",\n      "related": ["mcp-server", "external-tool", "guardrail"]\n    }\n  ]\n}\n', encoding='utf-8')
(SOURCE / 'scripts/make_sft_dataset.py').write_text('"""第2回 宿題（発展）: 用語集から SFT 用の Q&A データ（JSONL）を作る。\n\n使い方:\n    uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl\n\nできたファイルを Colab にアップロードし、LoRA 学習のデータとして使う。\n質問の言い回しを増やす、別名でも質問を作る、検証用に一部を分ける、などを工夫してみよう。\n"""\n\nimport json\nimport sys\nfrom pathlib import Path\n\nGLOSSARY = Path(__file__).resolve().parent.parent / "data" / "glossary.json"\nTEMPLATES = ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]\n\n\ndef main() -> None:\n    terms = json.loads(GLOSSARY.read_text(encoding="utf-8"))["terms"]\n    for t in terms:\n        for template in TEMPLATES:\n            record = {\n                "messages": [\n                    {"role": "user", "content": template.format(term=t["term"])},\n                    {"role": "assistant", "content": t["definition"]},\n                ]\n            }\n            sys.stdout.write(json.dumps(record, ensure_ascii=False) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

dataset_path = WORK / 'glossary_sft.jsonl'
with dataset_path.open('w', encoding='utf-8') as out:
    subprocess.run([sys.executable, str(SOURCE / 'scripts/make_sft_dataset.py')], stdout=out, check=True)
terms = json.loads((SOURCE / 'data/glossary.json').read_text())['terms']
by_id = {t['id']: t for t in terms}
train_records = [json.loads(line) for line in dataset_path.read_text().splitlines() if line.strip()]
valid_records = [{'messages': [
    {'role': 'user', 'content': f'用語「{t["term"]}」の意味を説明してください。'},
    {'role': 'assistant', 'content': t['definition']},
]} for t in terms]

# この評価セットは実行結果を見る前に固定。ORで許容する表現を並べる。
EVAL_SPECS = [
    ('lora', 'Low-Rank Adaptation', [['固定'], ['行列', '低ランク'], ['学習']]),
    ('qlora', 'Quantized LoRA', [['4', '四'], ['量子化'], ['LoRA']]),
    ('peft', 'Parameter-Efficient Fine-Tuning', [['一部', '少数', '小さな'], ['パラメータ'], ['学習']]),
    ('fine-tuning', '追加学習', [['学習済', '事前学習'], ['追加学習', '再学習', '調整']]),
    ('context-window', 'コンテキスト長', [['入力'], ['出力'], ['トークン'], ['上限', '最大']]),
    ('learning-rate', 'learning rate', [['パラメータ', '重み'], ['更新', '動かす', '変化']]),
    ('epoch', 'epoch', [['学習データ', 'データセット', 'データ全体'], ['1周', '一周', '1回', '一回']]),
    ('overfitting', 'オーバーフィッティング', [['学習データ', '訓練データ'], ['未知', '汎化', '新しい']]),
    ('homework-levels', '宿題レベル', [['基礎'], ['実装'], ['発展']]),
    ('answer-md', '解答ファイル', [['answer.md'], ['結果', '考察', '解答']]),
]
test = []
for term_id, alias, checks in EVAL_SPECS:
    t = by_id[term_id]
    for group, name, template in [
        ('known', t['term'], '「{}」の要点を説明してください。'),
        ('alias', alias, '「{}」を日本語で解説してください。'),
    ]:
        test.append({'group': group, 'term_id': term_id, 'query_term': name,
                     'question': template.format(name), 'reference': t['definition'], 'checks': checks})
for name, ref in [
    ('ドロップアウト', '実在する未収録語。学習時に一部のユニット等を確率的に無効化し、過学習を抑える手法。'),
    ('バッチ正規化', '実在する未収録語。バッチ内の平均や分散等を使って活性を正規化する手法。'),
]:
    test.append({'group': 'outside_real', 'term_id': None, 'query_term': name,
                 'question': f'「{name}」の意味を説明してください。', 'reference': ref,
                 'checks': None})
for name in ['AIE架空用語ゼータ17', 'AIE架空用語オメガ29']:
    test.append({'group': 'outside_invented', 'term_id': None, 'query_term': name,
                 'question': f'「{name}」の意味を説明してください。',
                 'reference': 'この実験で作成した架空語。定義を創作せず、不明・未登録として回答する。', 'checks': None})

def user_question(record):
    return next(m['content'] for m in record['messages'] if m['role'] == 'user')
train_q = {user_question(r) for r in train_records}
valid_q = {user_question(r) for r in valid_records}
test_q = {r['question'] for r in test}
assert not (train_q & valid_q or train_q & test_q or valid_q & test_q)
assert len(test_q) == len(test)
assert len(train_records) == 2 * len(terms)
(WORK / 'valid.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in valid_records), encoding='utf-8')
(WORK / 'eval_gold.json').write_text(json.dumps(test, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'用語={len(terms)} train={len(train_records)} valid={len(valid_records)} test={len(test)}')
print('質問重複なし。学習レコード例：', train_records[0])

用語=58 train=116 valid=58 test=24
質問重複なし。学習レコード例： {'messages': [{'role': 'user', 'content': '「AIエンジニアリング」とは何ですか？'}, {'role': 'assistant', 'content': '学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。'}]}


## 3. モデルの読み込み・学習前評価
モデルに渡すのは質問だけです。正解の定義や用語集全文は入力に含めません。
`point_match_rate` は既知語・別名問題の要点一致率です。正答率ではなく補助指標です。
用語集外の実在語と架空語は別々に見ます。確認要求率をそのまま正答率とはしません。

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


assert DEVICE == 'cuda', 'T4 GPUに切り替えてから、セクション0から実行してください。'
model.eval()

def generate(model, question):
    messages = [{'role': 'user', 'content': question}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
                                pad_token_id=tok.pad_token_id, **CONFIG['gen'])
    ids = output[0][inputs['input_ids'].shape[1]:]
    return tok.decode(ids, skip_special_tokens=True).strip(), len(ids)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [6]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

def normalize(text):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', text).casefold())

def point_match(text, checks):
    if checks is None:
        return None
    normalized = normalize(text)
    return all(any(normalize(word) in normalized for word in group) for group in checks)

def abstained(text):
    markers = ['わかりません', '分かりません', '不明', '確認が必要', '未登録', '見つかりません', '存じません', '定義されていません', '存在しません']
    return any(word in text for word in markers)

def summarize(rows, label):
    groups = []
    for group in dict.fromkeys(r['group'] for r in rows):
        subset = [r for r in rows if r['group'] == group]
        scored = [r['point_match'] for r in subset if r['point_match'] is not None]
        groups.append({'method': label, 'group': group, 'n': len(subset),
                       'point_match_rate': sum(scored)/len(scored) if scored else None,
                       'abstention_rate': sum(r['abstained'] for r in subset)/len(subset),
                       'seconds': round(sum(r['seconds'] for r in subset), 3)})
    return groups

def evaluate(model, label):
    rows = []
    for i, ex in enumerate(test, 1):
        start = time.perf_counter()
        answer, n_tokens = generate(model, ex['question'])
        rows.append({**ex, 'answer': answer, 'point_match': point_match(answer, ex['checks']),
                     'abstained': abstained(answer), 'generated_tokens': n_tokens,
                     'hit_output_limit': n_tokens >= CONFIG['max_new_tokens'],
                     'seconds': round(time.perf_counter() - start, 3)})
        print(f'{label} {i}/{len(test)} {ex["group"]}: {answer[:80]}')
    (WORK / f'eval_{label}.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    return rows

before_rows = evaluate(model, 'before')
display(pd.DataFrame(summarize(before_rows, 'before')))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


before 1/24 known: **LoRA（Low-Rank Adaptation）** は、大規模言語モデル（LLM）や他の大規模モデルに対して、**「少量の知識を、非常に大きなモデルに適
before 2/24 alias: **低ランク適応（Low-Rank Adaptation）** は、機械学習において「低ランク（低次元）のデータ」に対して、その特徴を**低ランク（低次元）のモ
before 3/24 known: **QLoRA** は、主に **Google DeepMind** が開発した、**大規模言語モデル（LLM）の最適化技術**です。従来の LLM（大規模言語
before 4/24 alias: **Quantized LoRA** は、大規模言語モデル（LLM）の学習や推理において、**「ローカル化（Localisation）」**された軽量モデルを、
before 5/24 known: **PEFT** は、大規模言語モデル（LLM）の学習において非常に重要な概念です。特に、**Prompt Engineering**（プロンプトエンジニアリン
before 6/24 alias: **パラメータ効率的なトレーニング（Parameter-Efficient Fine-Tuning, PEFT）** は、モデルの学習効率を最大化するために、ト
before 7/24 known: 「ファインチューニング」は、ソフトウェアやシステムにおいて、ユーザーの要望や実際の動作に合わせて、より良いパフォーマンスや安定性を高めるための**最終的な調整プ
before 8/24 alias: 「追加学習」は、学習の過程において**新しい知識やスキルを習得する**ことを指します。

一般的には、以下の 3 つの要素から構成されます：

1.  **新し
before 9/24 known: 「コンテキストウィンドウ」は、主に **Web 開発（特に React や Next.js）** や **AI 開発（特に LLM や Chatbot）** の
before 10/24 alias: 「コンテキスト長」とは、**「コンテキスト（背景や背景知識）」の長さ**を指します。

この概念は主に**AI（人工知能）の学習や生成において**、モデルが理解

,method,group,n,point_match_rate,abstention_rate,seconds
0,before,known,10,0.2,0.0,115.172
1,before,alias,10,0.0,0.0,106.519
2,before,outside_real,2,NaN,0.0,21.229
3,before,outside_invented,2,NaN,0.0,21.063


## 4. LoRA学習
公式スクリプトが作った `messages` 形式のレコードをそのまま使います。
検証は別質問の58件です。ここでは条件を固定して3エポック学習し、testは学習に使いません。
全文が最大長以内に収まることを先に確認し、最終loss・学習時間も保存します。

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
valid_ds = Dataset.from_list(valid_records)
lengths = [len(tok.apply_chat_template(r['messages'], tokenize=True, enable_thinking=False))
           for r in train_records + valid_records]
assert max(lengths) <= CONFIG['max_seq_len'], f'最長{max(lengths)}tokens。max_seq_lenを増やしてください。'
print('最長トークン数:', max(lengths))
model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
train_output = trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 3)
FINAL_LOGGED_LOSS = next((row['loss'] for row in reversed(trainer.state.log_history) if 'loss' in row), None)
TRAIN_METRICS = {'seconds': TRAIN_SECONDS, 'mean_train_loss': train_output.training_loss, 'last_logged_loss': FINAL_LOGGED_LOSS, 'steps': trainer.state.global_step, 'epoch': trainer.state.epoch, 'log_history': trainer.state.log_history}
(WORK / 'train_metrics.json').write_text(json.dumps(TRAIN_METRICS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

最長トークン数: 2


Tokenizing train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/116 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/58 [00:00<?, ? examples/s]

... (3 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.267213
20,2.222411
30,1.751089
40,1.306696


学習時間: 108 秒 | ステップ数: 45 | エポック: 3.0


## 5. 学習後評価と前後比較
学習前と同じ24問、同じ生成条件で評価します。
`hit_output_limit=True` の回答は途中で切れた可能性があります。設定を変えて比較するなら前後とも同じ上限で評価し直してください。

In [8]:
after_rows = evaluate(tuned, 'after')
comparison = pd.DataFrame(summarize(before_rows, 'before') + summarize(after_rows, 'after'))
display(comparison)
detail = pd.DataFrame([{'group': b['group'], 'question': b['question'], 'reference': b['reference'],
                       'before': b['answer'], 'after': a['answer'],
                       'before_point_match': b['point_match'], 'after_point_match': a['point_match'],
                       'before_limit': b['hit_output_limit'], 'after_limit': a['hit_output_limit']}
                      for b, a in zip(before_rows, after_rows)])
display(detail)
regressed = [b['question'] for b, a in zip(before_rows, after_rows)
             if b['point_match'] is True and a['point_match'] is False]
print('要点一致が退行した質問:', regressed)

after 1/24 known: 学習する重みを、入力変数の1%程度に留める小さな行列で表し、その重みを学習し、元の大きな行列を元の入力変数の1%程度に留める小さな行列に戻す。学習する重みは、入
after 2/24 alias: 同じ入力と出力の構造を、低ランク（低次元）のモデルで表し、重みやパラメータをその低ランクに絞って学習させる手法。学習するパラメータが少なくなるため、学習するデー
after 3/24 known: 大規模モデルを小さなメモリで扱うための技術。重みを低ランク（低精度）で表し、メモリを節約する。
after 4/24 alias: LoRAの重みをより小さなモデルに収めること。メモリと計算量が増えないため、軽量なモデルを学習できる。
after 5/24 known: モデルを小さなデータセットで学習させるための手法。重みを小さなベクトルで表し、後処理時に重みを戻す。メモリと計算量に制限があるときや、データ共有を許さない場合な
after 6/24 alias: モデルの重みを、必要なデータや入力・出力の範囲に絞って学習させること。重みを増やしすぎるとメモリや計算量が増え、学習するデータも増えすぎるため、学習するデータや
after 7/24 known: 入力データと、その入力と出力のペアを学習するモデルの追加パラメータ。質問応答の精度を上げるために使う。本講座ではLLMをテキストで扱うため、テキスト入力・出力の
after 8/24 alias: 学習データに追加して、モデルの表現をより広く、深く、あるいは特定の領域に焦点を当てられること。Pydantic AIでは Pydantic AI 拡張機能の追加
after 9/24 known: LLMが1回の对话で処理できる入力と出力の合計トークン数。多くのモデルは100〜200トークン程度で終了。
after 10/24 alias: LLMが処理するテキストの総字数。1回の質問応答では、質問と回答の合計字数が上限（例：128字）に収まらなければ回答が返らない。
after 11/24 known: 1回の更新でパラメータをどれだけ動かすかを決める超参数。大きすぎると学習が不安定になり、小さすぎると進みすぎない。
after 12/24 alias: 学習の進め方を決めるパラメータ。学習

,method,group,n,point_match_rate,abstention_rate,seconds
0,before,known,10,0.2,0.0,115.172
1,before,alias,10,0.0,0.0,106.519
2,before,outside_real,2,NaN,0.0,21.229
3,before,outside_invented,2,NaN,0.0,21.063
4,after,known,10,0.1,0.0,21.856
5,after,alias,10,0.3,0.0,25.454
6,after,outside_real,2,NaN,0.0,5.135
7,after,outside_invented,2,NaN,0.0,4.615


,group,question,reference,before,after,before_point_match,after_point_match,before_limit,after_limit
0,known,「LoRA」の要点を説明してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA（Low-Rank Adaptation）** は、大規模言語モデル（LLM）や他の大規模モデルに対して、**「少量の知識を、非常に大きなモデルに適用する」**技術です。\n\n従来の「ハイパーパラメータ調整（Hyperparameter Tuning）」や「モデル重み調整（Weight Tuning）」とは異なり、LoRA は学習プロセスを大きく変更し、モデルの性能を劇的に向上させる方法です。\n\n以下に LoRA の主なポイントを解説します。\n\n### 1. 核心となる概念：低ランク（Low-Rank）\nLoRA の最大の魅力は、**「低ランク（Low-Rank）」**という概念にあります。\n* **低ランク**とは、モデルの出力（ベクトル）やパラメータが、元のモデルよりも**小さく、単純な構造**であることを意味します。\n* 元のモデルには、数千万のパラメータ（約 100 億〜1000 億）が埋め込まれています。LoRA は、これらに「少量の知識」を埋め込むことで、元のモデルの複雑さや精度を維持しつつ、学習速度を大幅に向上させ,学習する重みを、入力変数の1%程度に留める小さな行列で表し、その重みを学習し、元の大きな行列を元の入力変数の1%程度に留める小さな行列に戻す。学習する重みは、入力変数の数よりも数値が小さい。,False,False,True,False
1,alias,「Low-Rank Adaptation」を日本語で解説してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**低ランク適応（Low-Rank Adaptation）** は、機械学習において「低ランク（低次元）のデータ」に対して、その特徴を**低ランク（低次元）のモデル**に適応させる手法を指します。\n\nこの手法は、従来の「高ランク（高次元）モデル」に適用するのではなく、データ自体の構造を低ランクに捉え、その低ランクモデルを用いて学習することで、より効率的に学習できることを目指します。\n\n### 主な特徴と目的\n\n1. **低ランクのデータ**:\n * 従来のデータは「高ランク（高次元）」であり、多くの特徴量（特徴ベクトル）を占めています。\n * 低ランクのデータは、少数の少数の要素（少数の特徴量）を占めており、計算コストが低く、学習速度が速いです。\n\n2. **低ランクのモデル**:\n * 従来のモデルは「高ランク（高次元）」で、多くのパラメータを占めています。\n * 低ランクのモデルは、少数の要素を占めるため、計算負荷が非常に小さくなります。\n\n3. **適応,同じ入力と出力の構造を、低ランク（低次元）のモデルで表し、重みやパラメータをその低ランクに絞って学習させる手法。学習するパラメータが少なくなるため、学習するデータや計算量も減る。,False,False,True,False
2,known,「QLoRA」の要点を説明してください。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,**QLoRA** は、主に **Google DeepMind** が開発した、**大規模言語モデル（LLM）の最適化技術**です。従来の LLM（大規模言語モデル）は、学習コストが高く、計算量も大きいため、特に**低コストで高速な処理**を実現するために、この技術が注目されています。\n\nQLoRA は、**「学習コストを削減しつつ、モデルの精度を維持する」**という二つの目標を達成するために設計されました。\n\n主な特徴とポイントについて解説します。\n\n### 1. 学習コストの大幅削減\n従来の LLM は、大量のデータ（数百万〜数十億もの単語）を学習する必要があるため、計算コストが非常に高くなります。\n* **QLoRA のアプローチ**: 学習コストを大幅に削減するために、**「学習率（Learning Rate）」を非常に小さく設定**します。\n* **効果**: 学習する単語の数が減り、計算量も減ります。これにより、**1 秒あたりの学習コストが従来の LLM の数倍に低下**します。\n\n### 2. 精度の維持と向上\n学習コストの削減は、モデルの表現能力（精度,大規模モデルを小さなメモリで扱うための技術。重みを低ランク（低精度）で表し、メモリを節約する。,False,False,True,False
3,alias,「Quantized LoRA」を日本語で解説してください。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,**Quantized LoRA** は、大規模言語モデル（LLM）の学習や推理において、**「ローカル化（Localisation）」**された軽量モデルを、**「量子化（Quantization）」**された形式で保存・動作させる技術です。\n\nこの技術は、大規模モデルの計算コストを大幅に削減し、推理速度を劇的に向上させるために開発されました。\n\n以下に詳細な解説をまとめます。\n\n### 1. 基本の仕組み\nLLM は、数百万のパラメータを持つ巨大なモデルです。これらをすべて計算して学習・推理するのは現実的ではありません。そのため、**「ローカル化（Localisation）」**という手法で、モデルの一部だけを計算し、残りのパラメータを「ローカル化（Quantization）」された形式で保存します。\n\n* **ローカル化（Localisation）**：モデルの大部分を計算し、残りのパラメータを保存。\n* **量子化（Quantization）**：保存されたパラメータを、計算コストを大幅に削減する形式に変換。\n\n### 2. 主な量子化技術\n現在の主流は以下の 2 つの技術です。\n\n#### A.,LoRAの重みをより小さなモデルに収めること。メモリと計算量が増えないため、軽量なモデルを学習できる。,False,False,True,False
4,known,「PEFT」の要点を説明してください。,モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。,**PEFT** は、大規模言語モデル（LLM）の学習において非常に重要な概念です。特に、**Prompt Engineering**（プロンプトエンジニアリング）において、ユーザーがどのように質問や指示を出すかによって、モデルがどのように回答するかが変わるという点に注目されています。\n\nPEFT は「**P**rompt **E**ngineering **T**raining**F**ormat（プロンプトエンジニアリングの学習フォーマット）」」を指します。\n\n以下に、PEFT の主なポイントと特徴を解説します。\n\n### 1. 核心となる概念：Prompt Engineering\nPEFT は、単に「質問を返す」だけでなく、**「質問をどう返すか」**という工程を重視しています。\n* **Prompt**: 人間が指示する言葉（例：「何を食べるべきか？」）\n* **Engineer**: 人間が指示する言葉の構造や形式を設計する技術者\n* **Training**: 学習するプロセス\n* **Format**: 学習する形式\n\nつまり、PEFT は「人間が何を指示するか（Prompt）」「その指示の形式（Format）」「その指示をどのように学習する（Training）」「,モデルを小さなデータセットで学習させるための手法。重みを小さなベクトルで表し、後処理時に重みを戻す。メモリと計算量に制限があるときや、データ共有を許さない場合などに使われる。,False,False,True,False
5,alias,「Parameter-Efficient Fine-Tuning」を日本語で解説してください。,モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる

要点一致が退行した質問: ['「過学習」の要点を説明してください。', '「answer.md」の要点を説明してください。']


## 6. 用語集を検索する方式との比較
引用符内の語を取り出し、同じ正規化をした正規名・別名に完全一致させます。
goldの `term_id` は検索に使いません。LoRAとQLoRAの部分一致も行いません。
これはLLMを使わない辞書検索であり、意味検索やRAG全体の性能を代表するものではありません。
検索は用語集を参照し、モデルは質問だけを受け取る、という方式の差も考察に含めます。

In [9]:
def search_glossary(question):
    match = re.search(r'「([^」]+)」', question)
    query = normalize(match.group(1) if match else question)
    hits = [t for t in terms if query in {normalize(name) for name in [t['term'], *t['aliases']]}]
    if len(hits) == 1:
        return hits[0]['definition']
    if len(hits) > 1:
        return '複数の項目に一致しました。確認が必要です。'
    return 'この用語集には未登録です。確認が必要です。'

retrieval_rows = []
for ex in test:
    start = time.perf_counter()
    answer = search_glossary(ex['question'])
    retrieval_rows.append({**ex, 'answer': answer, 'point_match': point_match(answer, ex['checks']),
                           'abstained': abstained(answer), 'seconds': time.perf_counter() - start})
comparison = pd.DataFrame(summarize(before_rows, 'before') + summarize(after_rows, 'after')
                          + summarize(retrieval_rows, 'dictionary_search'))
display(comparison)
manual = pd.DataFrame([{'group': b['group'], 'question': b['question'], 'reference': b['reference'],
                       'before': b['answer'], 'after': a['answer'], 'search': r['answer'],
                       'before_manual_0or1': '', 'after_manual_0or1': '', 'search_manual_0or1': '', 'notes': ''}
                      for b, a, r in zip(before_rows, after_rows, retrieval_rows)])
display(manual)
manual.to_csv(WORK / 'manual_review.csv', index=False, encoding='utf-8-sig')
(WORK / 'eval_search.json').write_text(json.dumps(retrieval_rows, ensure_ascii=False, indent=2), encoding='utf-8')
comparison.to_csv(WORK / 'comparison.csv', index=False, encoding='utf-8-sig')

,method,group,n,point_match_rate,abstention_rate,seconds
0,before,known,10,0.2,0.0,115.172
1,before,alias,10,0.0,0.0,106.519
2,before,outside_real,2,NaN,0.0,21.229
3,before,outside_invented,2,NaN,0.0,21.063
4,after,known,10,0.1,0.0,21.856
5,after,alias,10,0.3,0.0,25.454
6,after,outside_real,2,NaN,0.0,5.135
7,after,outside_invented,2,NaN,0.0,4.615
8,dictionary_search,known,10,1.0,0.0,0.006
9,dictionary_search,alias,10,1.0,0.0,0.005


,group,question,reference,before,after,search,before_manual_0or1,after_manual_0or1,search_manual_0or1,notes
0,known,「LoRA」の要点を説明してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA（Low-Rank Adaptation）** は、大規模言語モデル（LLM）や他の大規模モデルに対して、**「少量の知識を、非常に大きなモデルに適用する」**技術です。\n\n従来の「ハイパーパラメータ調整（Hyperparameter Tuning）」や「モデル重み調整（Weight Tuning）」とは異なり、LoRA は学習プロセスを大きく変更し、モデルの性能を劇的に向上させる方法です。\n\n以下に LoRA の主なポイントを解説します。\n\n### 1. 核心となる概念：低ランク（Low-Rank）\nLoRA の最大の魅力は、**「低ランク（Low-Rank）」**という概念にあります。\n* **低ランク**とは、モデルの出力（ベクトル）やパラメータが、元のモデルよりも**小さく、単純な構造**であることを意味します。\n* 元のモデルには、数千万のパラメータ（約 100 億〜1000 億）が埋め込まれています。LoRA は、これらに「少量の知識」を埋め込むことで、元のモデルの複雑さや精度を維持しつつ、学習速度を大幅に向上させ,学習する重みを、入力変数の1%程度に留める小さな行列で表し、その重みを学習し、元の大きな行列を元の入力変数の1%程度に留める小さな行列に戻す。学習する重みは、入力変数の数よりも数値が小さい。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,,,,
1,alias,「Low-Rank Adaptation」を日本語で解説してください。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**低ランク適応（Low-Rank Adaptation）** は、機械学習において「低ランク（低次元）のデータ」に対して、その特徴を**低ランク（低次元）のモデル**に適応させる手法を指します。\n\nこの手法は、従来の「高ランク（高次元）モデル」に適用するのではなく、データ自体の構造を低ランクに捉え、その低ランクモデルを用いて学習することで、より効率的に学習できることを目指します。\n\n### 主な特徴と目的\n\n1. **低ランクのデータ**:\n * 従来のデータは「高ランク（高次元）」であり、多くの特徴量（特徴ベクトル）を占めています。\n * 低ランクのデータは、少数の少数の要素（少数の特徴量）を占めており、計算コストが低く、学習速度が速いです。\n\n2. **低ランクのモデル**:\n * 従来のモデルは「高ランク（高次元）」で、多くのパラメータを占めています。\n * 低ランクのモデルは、少数の要素を占めるため、計算負荷が非常に小さくなります。\n\n3. **適応,同じ入力と出力の構造を、低ランク（低次元）のモデルで表し、重みやパラメータをその低ランクに絞って学習させる手法。学習するパラメータが少なくなるため、学習するデータや計算量も減る。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,,,,
2,known,「QLoRA」の要点を説明してください。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,**QLoRA** は、主に **Google DeepMind** が開発した、**大規模言語モデル（LLM）の最適化技術**です。従来の LLM（大規模言語モデル）は、学習コストが高く、計算量も大きいため、特に**低コストで高速な処理**を実現するために、この技術が注目されています。\n\nQLoRA は、**「学習コストを削減しつつ、モデルの精度を維持する」**という二つの目標を達成するために設計されました。\n\n主な特徴とポイントについて解説します。\n\n### 1. 学習コストの大幅削減\n従来の LLM は、大量のデータ（数百万〜数十億もの単語）を学習する必要があるため、計算コストが非常に高くなります。\n* **QLoRA のアプローチ**: 学習コストを大幅に削減するために、**「学習率（Learning Rate）」を非常に小さく設定**します。\n* **効果**: 学習する単語の数が減り、計算量も減ります。これにより、**1 秒あたりの学習コストが従来の LLM の数倍に低下**します。\n\n### 2. 精度の維持と向上\n学習コストの削減は、モデルの表現能力（精度,大規模モデルを小さなメモリで扱うための技術。重みを低ランク（低精度）で表し、メモリを節約する。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,,,,
3,alias,「Quantized LoRA」を日本語で解説してください。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,**Quantized LoRA** は、大規模言語モデル（LLM）の学習や推理において、**「ローカル化（Localisation）」**された軽量モデルを、**「量子化（Quantization）」**された形式で保存・動作させる技術です。\n\nこの技術は、大規模モデルの計算コストを大幅に削減し、推理速度を劇的に向上させるために開発されました。\n\n以下に詳細な解説をまとめます。\n\n### 1. 基本の仕組み\nLLM は、数百万のパラメータを持つ巨大なモデルです。これらをすべて計算して学習・推理するのは現実的ではありません。そのため、**「ローカル化（Localisation）」**という手法で、モデルの一部だけを計算し、残りのパラメータを「ローカル化（Quantization）」された形式で保存します。\n\n* **ローカル化（Localisation）**：モデルの大部分を計算し、残りのパラメータを保存。\n* **量子化（Quantization）**：保存されたパラメータを、計算コストを大幅に削減する形式に変換。\n\n### 2. 主な量子化技術\n現在の主流は以下の 2 つの技術です。\n\n#### A.,LoRAの重みをより小さなモデルに収めること。メモリと計算量が増えないため、軽量なモデルを学習できる。,ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。,,,,
4,known,「PEFT」の要点を説明してください。,モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。,**PEFT** は、大規模言語モデル（LLM）の学習において非常に重要な概念です。特に、**Prompt Engineering**（プロンプトエンジニアリング）において、ユーザーがどのように質問や指示を出すかによって、モデルがどのように回答するかが変わるという点に注目されています。\n\nPEFT は「**P**rompt **E**ngineering **T**raining**F**ormat（プロンプトエンジニアリングの学習フォーマット）」」を指します。\n\n以下に、PEFT の主なポイントと特徴を解説します。\n\n### 1. 核心となる概念：Prompt Engineering\nPEFT は、単に「質問を返す」だけでなく、**「質問をどう返すか」**という工程を重視しています。\

## 7. 結果の保存と考察
**このセルまで実行するとDriveへ結果とadapterを保存します。** Driveを使わない場合も、左のファイル一覧に出る `results.zip` をダウンロードできます。

保存した `manual_review.csv` で、意味の正しさ・説明不足・矛盾を確認し、0/1と理由を記入してください。
実在未収録語は正しく説明していればそれも記録します。架空語の定義を作った場合とは分けます。

`answer.md` に書く観点：
1. 用語・質問数、分割方法、モデル・設定、自分が実行した範囲。
2. 学習前後で変わった回答・退行・別名への対応。
3. 未収録語と架空語で、正しい説明／確認要求／創作がどう分かれたか。
4. 辞書検索と比べた正確さ、待ち時間、更新の容易さ、学習・推論のコスト。

共通演習のFAQ実験は元のノートブックにあります。この発展版は、宿題の発展部分に対応します。
最後に出力を含むipynbも保存し、GPUランタイムを削除してください。

In [10]:
result = {
    'assignment': 'lecture02_advanced_glossary', 'run_id': RUN_ID,
    'config': CONFIG, 'executed_training_myself': True,
    'source_blobs': {'glossary': 'bc0d6807bdb9aeef7ef9eb98e74d767e056ed745',
                     'generator': '7e53cfc38f0b70f0141b6f98eab18a94c16a2645'},
    'versions': {module.__name__: module.__version__ for module in [transformers, peft, trl, torch]},
    'counts': {'terms': len(terms), 'train': len(train_records), 'valid': len(valid_records), 'test': len(test)},
    'training': TRAIN_METRICS,
    'summaries': summarize(before_rows, 'before') + summarize(after_rows, 'after') + summarize(retrieval_rows, 'dictionary_search'),
    'regressed': regressed,
    'evaluation_note': '要点一致率は補助指標。全文の目視判定はmanual_review.csvに別記。',
}
(WORK / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
# adapterとチェックポイントを除く、提出・分析用の軽量ファイルをまとめる。
archive = WORK / 'export'
archive.mkdir(exist_ok=True)
for path in list(WORK.iterdir()):
    if path.is_file() and path.suffix in {'.json', '.jsonl', '.csv'}:
        shutil.copy2(path, archive / path.name)
shutil.copytree(SOURCE, archive / 'source', dirs_exist_ok=True)
zip_path = shutil.make_archive(str(WORK / 'results'), 'zip', archive)
if USE_DRIVE:
    destination = Path(CONFIG['drive_dir'])
    for path in archive.iterdir():
        if path.is_file():
            shutil.copy2(path, destination / path.name)
    shutil.copytree(SOURCE, destination / 'source', dirs_exist_ok=True)
    shutil.copy2(zip_path, destination / 'results.zip')
    print('Drive保存先:', destination)
print('結果ZIP:', zip_path)
print('学習時間:', TRAIN_METRICS['seconds'], '秒 / 最終記録loss:', TRAIN_METRICS['last_logged_loss'])

Drive保存先: /content/drive/MyDrive/aie2026/lecture02_advanced/20261102_160734_926704
結果ZIP: /content/lecture02_advanced/20261102_160734_926704/results.zip
学習時間: 107.784 秒 / 最終記録loss: 1.3066959381103516
